# Phase 2B — YOLOv8n Eye-Shield Baseline Training (Google Colab)

Two-class compliance detector: **0 `goggles`**, **1 `no_goggles`**.

**Read this before running anything.**

* The dataset is the approved baseline at `data/processed/eye_shield_baseline/`
  (240 images: 160 train / 40 val / 40 test, balanced 120/120).
* The **test split is locked**. It must not influence training, early stopping,
  confidence-threshold selection, augmentation choice, or hyper-parameters.
  Cell 6 writes a training YAML that does not mention the test split at all, so
  the trainer structurally cannot see it.
* The test split is evaluated **once**, in Cell 9, after training is finished.
* **No metric in this notebook is pre-filled.** Every number you see is
  produced by your own run. Do not report numbers you did not measure.

**Runtime requirements:** `Runtime → Change runtime type → T4 GPU` (or better).

## 1. Install a pinned Ultralytics version

Pinned to the version verified locally so the reported metrics are attributable to a known stack.

In [ ]:
# Pinned stack. Ultralytics pulls a compatible torch/torchvision for the Colab GPU.
PINNED_ULTRALYTICS = "8.4.215"

!pip install -q "ultralytics=={PINNED_ULTRALYTICS}"

## 2. Print the environment

Recorded so every metric can be traced to the exact software stack and GPU.

In [ ]:
import json, platform, sys

import torch
import ultralytics

env = {
    "python": sys.version.split()[0],
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ) if torch.cuda.is_available() else None,
    "platform": platform.platform(),
}

print(json.dumps(env, indent=2))

if not torch.cuda.is_available():
    print("\n*** WARNING: no CUDA GPU. Enable Runtime -> Change runtime type -> T4 GPU. ***")
    print("*** Training on CPU here will be very slow. ***")

## 3. Upload the dataset ZIP

Build the ZIP on your Mac first (the dataset images and labels are Git-ignored,
so they are not in the repository):

```bash
cd /Users/hermanyeung/Downloads/AIOT-MiniProject
rm -f /tmp/eye_shield_baseline.zip
cd data/processed && zip -qr /tmp/eye_shield_baseline.zip eye_shield_baseline \
  -x 'eye_shield_baseline/artifacts/*' && cd -
ls -lh /tmp/eye_shield_baseline.zip
```

Then run this cell and pick `/tmp/eye_shield_baseline.zip`.

The ZIP must contain `eye_shield_baseline/data.yaml` at that path (i.e. the
archive root holds one directory called `eye_shield_baseline`).

In [ ]:
from google.colab import files

uploaded = files.upload()   # choose eye_shield_baseline.zip
print("uploaded:", list(uploaded))

## 4. Extract the dataset

In [ ]:
import os, shutil, zipfile
from pathlib import Path

WORK = Path("/content/eye_shield")
WORK.mkdir(parents=True, exist_ok=True)

zips = [p for p in Path("/content").glob("*.zip")]
assert zips, "No .zip found in /content — run the upload cell first."
src_zip = sorted(zips, key=lambda p: p.stat().st_mtime)[-1]
print("extracting:", src_zip)

with zipfile.ZipFile(src_zip) as zf:
    zf.extractall(WORK)

# Locate the directory that actually contains data.yaml.
candidates = list(WORK.rglob("data.yaml"))
assert candidates, f"no data.yaml under {WORK}"
DATASET_ROOT = candidates[0].parent
print("dataset root:", DATASET_ROOT)

## 5. Validate `data.yaml` and both class names

Aborts if the class contract does not match `config/classes.yaml` (`0 goggles`, `1 no_goggles`).

In [ ]:
import yaml

EXPECTED_CLASSES = {0: "goggles", 1: "no_goggles"}

yaml_path = DATASET_ROOT / "data.yaml"
cfg = yaml.safe_load(yaml_path.read_text())

names = cfg.get("names")
if isinstance(names, dict):
    names = {int(k): str(v) for k, v in names.items()}
else:
    names = {i: str(v) for i, v in enumerate(names)}

print("raw data.yaml:")
print(yaml_path.read_text())

assert names == EXPECTED_CLASSES, f"class mismatch: {names} != {EXPECTED_CLASSES}"
print("\nOK — classes are exactly:", names)

## 6. Count the splits and build the training-only YAML

Cell writes `/content/train_only_data.yaml`, which contains **no `test:` key**. That is what keeps the locked split out of training.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def count_images(split):
    d = DATASET_ROOT / split / "images"
    return sum(1 for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)

def count_labels(split):
    d = DATASET_ROOT / split / "labels"
    return sum(1 for p in d.iterdir() if p.is_file() and p.suffix.lower() == ".txt")

counts = {}
for split in ("train", "val", "test"):
    n_img, n_lbl = count_images(split), count_labels(split)
    counts[split] = {"images": n_img, "labels": n_lbl}
    print(f"{split:<6} {n_img:>4} images  {n_lbl:>4} labels")
    assert n_img > 0, f"split {split} is empty"
    assert n_img == n_lbl, f"split {split} image/label mismatch"

TRAIN_YAML = Path("/content/train_only_data.yaml")
TRAIN_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "names": dict(EXPECTED_CLASSES),
}, sort_keys=False))

print("\ntraining YAML (note: no test split):")
print(TRAIN_YAML.read_text())
assert "test" not in TRAIN_YAML.read_text(), "test split leaked into the training YAML"

## 7. Train YOLOv8n

* `yolov8n.pt` pretrained COCO weights
* `imgsz=640`, `epochs=60` max, `patience=10` early stopping
* `seed=42`, `deterministic=True`
* `split="val"` — validation-only, never test
* Batch selected from GPU memory below; override `BATCH` if you hit an OOM.

In [ ]:
from ultralytics import YOLO

SEED = 42
IMGSZ = 640
EPOCHS = 60
PATIENCE = 10

# Conservative batch by GPU memory (Colab free tier is usually a 16 GB T4).
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    BATCH = 32 if gb >= 14 else (16 if gb >= 7 else 8)
    print(f"GPU memory: {gb:.2f} GB -> batch {BATCH}")
else:
    BATCH = 8
    print("no CUDA GPU -> batch", BATCH)

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(TRAIN_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    deterministic=True,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/content/runs",
    name="yolov8n_baseline",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    workers=2,
    cache=False,
    plots=True,
    save=True,
    val=True,
    split="val",       # validation only — the test split is not named anywhere
    verbose=True,
)

RUN_DIR = Path(model.trainer.save_dir)
print("\nrun dir:", RUN_DIR)

## 8. Confirm `best.pt` / `last.pt` and the training artifacts exist

In [ ]:
import shutil

for name in ("weights/best.pt", "weights/last.pt", "results.csv", "results.png",
             "confusion_matrix.png", "confusion_matrix_normalized.png"):
    p = RUN_DIR / name
    print(f"{'OK ' if p.exists() else 'MISSING ':<8} {name}"
          + (f"  ({p.stat().st_size/1024:.0f} KiB)" if p.exists() else ""))

BEST = RUN_DIR / "weights" / "best.pt"
LAST = RUN_DIR / "weights" / "last.pt"
assert BEST.exists(), "best.pt not found — training may have failed"
assert LAST.exists(), "last.pt not found — training may have failed"

# The per-epoch curve data. Copied verbatim; never edited.
results_csv_text = (RUN_DIR / "results.csv").read_text()
print("\nresults.csv columns:")
print(results_csv_text.splitlines()[0])
print(f"{len(results_csv_text.splitlines()) - 1} epoch rows")

## 9. Evaluate `best.pt` **once** on the locked test split

This is the only cell that touches the test split. Run it after training is final.
The on-disk dataset is not modified: an evaluation YAML is written whose `val`
entry points at `test/images`, because `model.val()` always reads the split named
`val`.

In [ ]:
EVAL_YAML = Path("/content/eval_test_split.yaml")
EVAL_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "test/images",     # <-- locked test split, read once
    "test": "test/images",
    "names": dict(EXPECTED_CLASSES),
}, sort_keys=False))

eval_model = YOLO(str(BEST))
model_names = {int(k): str(v) for k, v in eval_model.names.items()}
assert model_names == EXPECTED_CLASSES, f"weights class mismatch: {model_names}"

test_results = eval_model.val(
    data=str(EVAL_YAML),
    split="val",          # resolves to test/images via EVAL_YAML
    imgsz=IMGSZ,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True,
    save_json=False,
    project="/content/runs",
    name="locked_test_eval",
    exist_ok=True,
    verbose=False,
)
print("evaluation complete")

## 10. Export per-class and aggregate metrics

Every value below is read out of the evaluation object. All arithmetic is F1 from measured P and R — no targets, no estimates.

In [ ]:
box = test_results.box
present = list(box.ap_class_index)

per_class = {}
for pos, class_index in enumerate(present):
    class_index = int(class_index)
    p, r = float(box.p[pos]), float(box.r[pos])
    per_class[model_names[class_index]] = {
        "class_id": class_index,
        "precision": round(p, 6),
        "recall": round(r, 6),
        "f1": round(0.0 if p + r == 0 else 2 * p * r / (p + r), 6),
        "map50": round(float(box.ap50[pos]), 6),
        "map50_95": round(float(box.ap[pos]), 6),
    }

mean = box.mean_results()
overall = {
    "precision": round(float(mean[0]), 6),
    "recall": round(float(mean[1]), 6),
    "map50": round(float(mean[2]), 6),
    "map50_95": round(float(mean[3]), 6),
}
overall["f1"] = round(
    0.0 if overall["precision"] + overall["recall"] == 0
    else 2 * overall["precision"] * overall["recall"]
    / (overall["precision"] + overall["recall"]), 6
)

speed = dict(test_results.speed)
total_ms = sum(float(speed.get(k, 0.0)) for k in ("preprocess", "inference", "postprocess"))
speed_report = {
    "preprocess_ms": round(float(speed.get("preprocess", 0.0)), 3),
    "inference_ms": round(float(speed.get("inference", 0.0)), 3),
    "postprocess_ms": round(float(speed.get("postprocess", 0.0)), 3),
    "total_ms_per_image": round(total_ms, 3),
    "fps": round(1000.0 / total_ms, 3) if total_ms > 0 else None,
}

print(f"{'class':<14}{'P':>9}{'R':>9}{'F1':>9}{'mAP50':>9}{'mAP50-95':>11}")
for name, v in per_class.items():
    print(f"{name:<14}{v['precision']:>9.4f}{v['recall']:>9.4f}{v['f1']:>9.4f}"
          f"{v['map50']:>9.4f}{v['map50_95']:>11.4f}")
print(f"{'ALL':<14}{overall['precision']:>9.4f}{overall['recall']:>9.4f}"
      f"{overall['f1']:>9.4f}{overall['map50']:>9.4f}{overall['map50_95']:>11.4f}")
print("\nspeed:", speed_report)

## 11. Write the metrics JSON and the confusion matrix

In [ ]:
confusion = None
try:
    confusion = [[float(c) for c in row] for row in test_results.confusion_matrix.matrix]
except Exception as exc:
    print("confusion matrix unavailable:", exc)

report = {
    "evaluated_at": __import__("datetime").datetime.now(
        __import__("datetime").timezone.utc).isoformat(),
    "weights": str(BEST),
    "dataset_root": str(DATASET_ROOT),
    "test_split": "test",
    "test_images": counts["test"]["images"],
    "split_counts": counts,
    "imgsz": IMGSZ,
    "batch": 16,
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
    "classes": {str(k): v for k, v in model_names.items()},
    "metrics": {
        "per_class": per_class,
        "overall": overall,
        "speed": speed_report,
    },
    "confusion_matrix": confusion,
    "versions": env,
    "caveat": (
        "Measured once on the locked test split. The test split was not used for "
        "training, early stopping, threshold selection, augmentation selection, "
        "or hyper-parameter tuning."
    ),
}

REPORT_PATH = Path("/content/model_eval_test.json")
REPORT_PATH.write_text(json.dumps(report, indent=2))
print(REPORT_PATH.read_text()[:1200])

## 12. Package `best.pt` and every artifact for download

Then download `eye_shield_results.zip` and keep it — it is the only evidence
that training actually ran.

In [ ]:
import shutil

BUNDLE = Path("/content/eye_shield_results")
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)

# Weights
(BUNDLE / "weights").mkdir()
shutil.copy2(BEST, BUNDLE / "weights" / "best.pt")
shutil.copy2(LAST, BUNDLE / "weights" / "last.pt")

# Curves, confusion matrix, PR curves and any other plot Ultralytics produced
for pattern in ("*.png", "*.jpg", "*.csv", "*.yaml", "*.txt", "*.json"):
    for p in RUN_DIR.glob(pattern):
        shutil.copy2(p, BUNDLE / p.name)

# Locked-test evaluation artifacts
eval_dir = Path("/content/runs/locked_test_eval")
if eval_dir.exists():
    (BUNDLE / "test_eval").mkdir()
    for pattern in ("*.png", "*.jpg", "*.csv", "*.json"):
        for p in eval_dir.glob(pattern):
            shutil.copy2(p, BUNDLE / "test_eval" / p.name)

shutil.copy2(REPORT_PATH, BUNDLE / "model_eval_test.json")
(BUNDLE / "environment.json").write_text(json.dumps(env, indent=2))

zip_base = "/content/eye_shield_results"
shutil.make_archive(zip_base, "zip", BUNDLE)
print("\nbundle contents:")
for p in sorted(BUNDLE.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(BUNDLE)}  ({p.stat().st_size/1024:.0f} KiB)")

files.download(zip_base + ".zip")

## 13. Checklist before you report any result

- [ ] `best.pt` and `last.pt` downloaded.
- [ ] `results.csv`, `results.png`, `confusion_matrix.png` and the PR curves in the bundle.
- [ ] `model_eval_test.json` present — this is the source of every number you quote.
- [ ] You state the GPU the run used.
- [ ] You did **not** re-run training after looking at test metrics. If you tune and re-train, the test split is no longer locked and must be reported as contaminated.
- [ ] Copy `best.pt` into `models/best.pt` on your Mac (Git-ignored — never commit it).
- [ ] Append the metrics with `python training/export_metrics.py --report model_eval_test.json`.